# Nhánh 1 — So sánh kiến trúc

So sánh 4 candidate cho bộ phân loại SQLi đa lớp, chọn theo **F1-macro vs latency vs size** (không mặc định chọn transformer).

Dữ liệu: `data/processed/branch1_train.csv` (**67.796 dòng, 5 lớp**, train/test 54.236/13.560). Lớp `stacked` đã loại; và (20/8) dataset được **dedup `query_canonical` trước khi split** để bỏ cross-split leakage (audit Mảng 2: 949 text trùng cả train & test) — nên các con số dưới đây là **leakage-free**. CNN/DistilBERT có head đúng 5 lớp (suy ra từ dữ liệu).

Kết quả số liệu đọc từ `report/metrics/branch1_architecture_comparison.json` (sinh bởi `train/compare_branch1_architectures.py`).

In [1]:
import json
import pandas as pd

with open('../../report/metrics/branch1_architecture_comparison.json') as f:
    results = json.load(f)

rows = []
for name, m in results.items():
    rows.append({
        'model': name,
        'F1_macro': round(m['f1_macro'], 4),
        'p50_ms': round(m['latency']['p50_ms'], 3),
        'p95_ms': round(m['latency']['p95_ms'], 3),
        'train_s': round(m['train_time_s'], 1),
        'size_MB': round(m['model_size_bytes']/1024/1024, 2),
    })
df = pd.DataFrame(rows)
df

,model,F1_macro,p50_ms,p95_ms,train_s,size_MB
0,tfidf_logreg,0.9907,0.523,0.744,13.2,3.51
1,tfidf_lightgbm,0.9977,60.404,65.485,195.0,5.64
2,distilbert,0.9957,2.802,4.244,1498.6,256.11
3,cnn_sqltok,0.9906,0.349,0.387,10.0,0.11


## Per-class F1

In [2]:
classes = ['normal', 'union_based', 'error_based', 'boolean_blind', 'time_blind']
per_class = {name: {c: round(m['per_class'][c]['f1-score'], 3) for c in classes} for name, m in results.items()}
pd.DataFrame(per_class).T

,normal,union_based,error_based,boolean_blind,time_blind
tfidf_logreg,0.977,0.996,0.999,0.981,1.000
tfidf_lightgbm,0.994,1.000,1.000,0.994,1.000
distilbert,0.990,0.999,1.000,0.990,0.999
cnn_sqltok,0.991,0.996,1.000,0.978,0.988


## Nhận xét & Quyết định

Số liệu 5 lớp, **split sạch (đã dedup, leakage-free)**, test 13.560 dòng:

| Model | F1-macro | p50 latency | Size | Ghi chú |
|---|---|---|---|---|
| TF-IDF + LogReg | 0.9907 | 0.52ms | 3.5MB | Nhanh, đơn giản |
| TF-IDF + LightGBM | 0.9977 | 60.4ms | 5.6MB | F1 cao nhất **nhưng latency ~60ms** (chậm gấp ~115x) |
| DistilBERT | 0.9957 | 2.80ms | 256MB | F1 cao, latency ổn (GPU), **size lớn + train ~25 phút** |
| CNN + SQL-tokenizer | 0.9906 | 0.35ms | 0.11MB | **Nhanh nhất, nhỏ nhất (28.453 params)**, F1 sát nhóm dẫn đầu |

**Quyết định: chọn TF-IDF + LogReg** làm baseline production cho Nhánh 1.
- Chênh lệch F1 giữa 4 model rất nhỏ (0.991–0.998) → không đáng đánh đổi.
- LightGBM tuy F1 cao nhất nhưng latency ~60ms là quá cao cho database proxy real-time.
- DistilBERT không cho lợi ích F1 rõ rệt mà tốn 256MB + cần GPU + train lâu.
- CNN là ứng viên thay thế tốt (nhanh/nhỏ nhất) nếu cần thêm khả năng học đặc trưng.

> Weights CNN + DistilBERT (5 lớp, split sạch) trên HF: `Jason-42195/VNU-SQLi-Detection-Models`, thư mục `branch1_comparison/`.

> **Lưu ý (20/8):** so với bản split cũ (còn leakage) F1 **tăng nhẹ** ở cả 4 model (vd tfidf 0.982→0.991) — không phải do leakage (đã bỏ) mà do dedup giúp undersample lấy 15k dòng **distinct**/lớp từ pool lớn hơn, data đa dạng hơn.

⚠️ **Cảnh báo:** F1 cao đồng loạt (~0.99) → dữ liệu vẫn **dễ phân biệt**, chưa phải benchmark adversarial/obfuscation. Content-format duplication `/blog` ~17.6% (audit Mảng 4) vẫn làm giảm độ đa dạng per-class.